In [9]:
import requests
import os
from selenium import webdriver
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import re
from selenium.webdriver.chrome.options import Options
import time
reg_title = re.compile('【(.*?)】')
reg_unfold =re.compile( '展开')
url  = 'https://weibo.com/u/2803301701'
headers = {
    'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Cookie':'SUB=_2AkMS-WPBf8NxqwFRmfoRxGzibo9zyQvEieKkpZIaJRMxHRl-yT9kqkxdtRB6OXlNLnasnhc3GMLU1lxeZgVSuqQcvtNF; SUBP=0033WrSXqPxfM72-Ws9jqgMF55529P9D9WW.zgb4bVoLSbXXhE74YvNV; SINAGLOBAL=4906935630120.408.1705372918261; ULV=1705372918271:1:1:1:4906935630120.408.1705372918261:; XSRF-TOKEN=awOd4CdIjsRyTWJGXLrDgzYd; WBPSESS=V0zdZ7jH8_6F0CA8c_usscKbaaShoq4bbvaI39whcywCuZVfeCDAnQ9l_KtVjEdg4O2dctNJBv9MZFukVAqZL9ukf72TnQ3H1f3zQosa-UlNRg4jZC8swo0GjD2bY4uMEwxJelI6qfoneLaNMk1FfYFFKsh23ySnHeL_NRvQ7fc=',
    'Referer':'https://weibo.com/'
}
def find_numble_period(title,data):
    n = data.shape[0]
    if(n>5):
        n = 5
    for i in range(n):
        if data.iloc[i,1] == title:
            print(title,'  这个标签的视频已经爬取了')
            return -1
    k = data.shape[0]+1
    return k
def down_video(title,video):
    response = requests.get(url = video,headers = headers)
    with open(title+'.mp4','wb+') as fp:
        fp.write(response.content)
    print(title,'视频已经下载')

In [24]:
print('正在打开人民日报网站')
chrome_options = Options()
# chrome_options.add_argument("--headless")
# chrome_options.add_argument("--disable-gpu")
browser = webdriver.Chrome(options= chrome_options)
browser.get(url)
data = pd.read_excel("C:\\Users\\86139\\Desktop\\人民日报数据处理\\人民日报数据的收集.xlsx")
WebDriverWait(browser,1000).until(EC.presence_of_element_located((By.XPATH,'//*[@id="scroller"]')))
print('进入页面成功')
wait = WebDriverWait(browser,5)
for a in range(1,6):
    print(f'正在爬取第{a}篇文章')
    if a==1:
        browser.execute_script("window.scrollBy(0,500);")
        time.sleep(5)
    if a==2:
        browser.execute_script("window.scrollBy(0,700);")
        time.sleep(5) 
    if a==3:
        browser.execute_script("window.scrollBy(0,500);")
        time.sleep(5)
    if a==4:
        browser.execute_script("window.scrollBy(0,700);")
        time.sleep(5)
    if a==5:
        browser.execute_script("window.scrollBy(0,500);")
        time.sleep(5)
    xpath1 = '//*[@id="scroller"]/div[1]/div[{}]/div/article/div/div/div[1]/div'.format(a)
    try:
        text = wait.until(EC.visibility_of_element_located((By.XPATH,xpath1)))
        text = text.text
        judge_text = reg_unfold.findall(text)
        if judge_text != []:
            print("这个视频有展开")
            xpath_unfold = '//*[@id="scroller"]/div[1]/div[{}]/div/article/div/div/div[1]/div/span'.format(a)
            browser.find_element(By.XPATH,xpath_unfold).click()
            time.sleep(2)
            text = browser.find_element(By.XPATH,xpath1)
            text = text.text
            text = text.split('收起')[0]
        title = reg_title.findall(text)[0]
        content = text.split('】')[1]
        numble_period = find_numble_period(title,data)
        if numble_period != -1:
            xpath2 = '//*[@id="scroller"]/div[1]/div[{}]/div/article/div/div/div[2]/div/div/div/div/div/div/video'.format(a)
            try:
                video = wait.until(EC.visibility_of_element_located((By.XPATH,xpath2)))
                video = video.get_attribute('src')
                total = [numble_period,title,content,video]
                data = pd.concat([pd.DataFrame(total,index = ['期数','标题','内容','视频'],).T,data])
                data = data.reset_index(drop = True)
                path = "C:\\Users\\86139\\Desktop\\人民日报数据"
                os.chdir(path)
                file_name = str(numble_period) + '_' + title
                os.mkdir(file_name)
                path1  = os.path.join(path,file_name)
                os.chdir(path1)
                down_video(title,video)
            except TimeoutException:
                print(title,'     这个标签下没有视频')
    except TimeoutException:
        browser.close()
        print('本次爬取视频任务结束')
        break
print('浏览器关闭')
# browser.close()
os.chdir("C:\\Users\\86139\\Desktop\\人民日报数据处理")
data.to_excel('人民日报数据的收集.xlsx',index = False)
print('数据更新完毕')

正在打开人民日报网站
进入页面成功
正在爬取第1篇文章
这个视频有展开
【出行注意！#2010年以来首次冰冻橙色预警#发布】#7省市部分地区将有持续性冻雨# 2月1日10时，中央气象台首次发布冰冻橙色预警，这是自2010年设立冰冻预警标准以来发布的最高级别冰冻预警。2月1日至4日，河南南部、湖北、安徽中北部、江苏北部、湖南中北部、贵州东部、重庆东南部等地部分地区有持续性冻雨。气象专家提示，冻雨对交通运输、电力设施、公众出行等有较大影响。出行安全提醒↓↓转需！（人民日报记者李红梅）收起
出行注意！#2010年以来首次冰冻橙色预警#发布      这个标签下没有视频
正在爬取第2篇文章
这个视频有展开
#南极眼#【一起围观！#南极考察大洋作业都在干些啥#】中国第40次南极科学考察队依托“雪龙”号科考船，计划自1月下旬至3月上旬，在西南极的南极半岛临近海域，以及东南极的宇航员海、普里兹湾开展生物生态、水体环境、沉积环境、大气环境及污染物分布综合调查监测。作业期间，队员们不分昼夜，全天候、不间断进行调查。自当地时间1月23日“雪龙”号正式开展大洋作业以来，考察队已经完成了南极半岛临近海域的调查，目前正在前往宇航员海的途中。此前，中国第40次南极考察“雪龙2”号大洋队已于北京时间1月22日完成了大洋调查任务。（胡润新 刘诗瑶）收起
一起围观！#南极考察大洋作业都在干些啥#      这个标签下没有视频
正在爬取第3篇文章
神奇！#新疆一湖面如梦幻冰上森林#   这个标签的视频已经爬取了
正在爬取第4篇文章
收藏！#一个动作帮助改善弯腰驼背#   这个标签的视频已经爬取了
正在爬取第5篇文章
这个视频有展开
【#神秘人10年捐赠1000万元善款#】近日，浙江嘉兴海盐县慈善总会又收到神秘人“金粟缘人”捐款的100万元。从2015年到2024年，他每年都会打来100万元善款，总计已捐款1000万元，这些钱帮助了很多困难家庭。“金粟缘人”一直没透露自己的真实身份，每一次表彰会现场座位都是空的。善良会传递更多善良，谢谢你！（嘉兴日报社读嘉新闻）网页链接 人民日报的微博视频收起
#神秘人10年捐赠1000万元善款#   这个标签的视频已经爬取了
浏览器关闭


PermissionError: [Errno 13] Permission denied: '人民日报数据的收集.xlsx'

In [5]:
os.chdir("C:\\Users\\86139\\Desktop\\人民日报数据")
data.to_excel('人民日报数据的收集.xlsx',index = False)
print('数据更新完毕')

数据更新完毕


In [31]:
len(text)

183